### Dependencies and setup

In [24]:
from typing import TypedDict, List, Dict, Any, Annotated
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
import uuid
import random
from collections import Counter
from operator import add

### Define the state

In [25]:



class State(TypedDict):
  """State for the SDG workflow"""
  documents: List[Document]
  questions: Annotated[List[Dict[str, Any]], add]
  answers: Annotated[List[Dict[str, Any]], add]
  contexts: Annotated[List[Dict[str, Any]], add]
  target_count: int
  current_count: int
  llm: Any

### Helpers, evolution functions

##### requirement 1: simple evolution

In [26]:
def simple_evolution(state: State, llm) -> State:
  """Create simple question from a document"""
  doc = random.choice(state["documents"])
  context = doc.page_content

  prompt = ChatPromptTemplate.from_template("""
  Create a clear, direct question that can be answered from this context:
                                            
  Context: {context}
                                            
  Question:
  """)
  
  question = llm.invoke(prompt.format_messages(context=context)).content.strip()
  question_id = str(uuid.uuid4())

  new_question = {
    "id": question_id,
    "question": question,
    "evolution_type": "simple",
  }

  new_context = {
    "question_id": question_id,
    "contexts": [context],
  }

  return {
    "questions": [new_question],
    "contexts": [new_context],
    "current_count": state["current_count"] + 1,
  }

##### requirement 2: multi-context evolution

In [27]:
def multi_context_evolution(state: State, llm) -> State:
    """Create question requiring multiple contexts"""
    docs = random.sample(state["documents"], min(2, len(state["documents"])))
    contexts = [doc.page_content for doc in docs]
    
    prompt = ChatPromptTemplate.from_template("""
    Create a question that requires information from both contexts:
    
    Context 1: {context1}
    Context 2: {context2}
    
    Question:
    """)
    
    question = llm.invoke(prompt.format_messages(
        context1=contexts[0], 
        context2=contexts[1]
    )).content.strip()
    
    question_id = str(uuid.uuid4())
    
    new_question = {
        "id": question_id,
        "question": question,
        "evolution_type": "multi_context"
    }
    
    new_context = {
        "question_id": question_id,
        "contexts": contexts
    }
    
    return {
        "questions": [new_question],
        "contexts": [new_context],
        "current_count": state["current_count"] + 1
    }

##### requirement 3: reasoning evolution

In [28]:
def reasoning_evolution(state: State, llm) -> State:
    """Create question requiring reasoning"""
    doc = random.choice(state["documents"])
    context = doc.page_content
    
    prompt = ChatPromptTemplate.from_template("""
    Create a question that requires reasoning and inference from this context:
    
    Context: {context}
    
    Question:""")
    
    question = llm.invoke(prompt.format_messages(context=context)).content.strip()
    question_id = str(uuid.uuid4())
    
    new_question = {
        "id": question_id,
        "question": question,
        "evolution_type": "reasoning"
    }
    
    new_context = {
        "question_id": question_id,
        "contexts": [context]
    }
    
    return {
        "questions": [new_question],
        "contexts": [new_context],
        "current_count": state["current_count"] + 1
    }

##### Now, let's define a function that takes the most recent question and generates an answer for it.

In [29]:
def generate_answer(state: State, llm) -> State:
    """Generate answer for the most recent question"""
    latest_question = state["questions"][-1]
    latest_context = state["contexts"][-1]
    
    prompt = ChatPromptTemplate.from_template("""
    Answer this question based on the provided context(s):
    
    Question: {question}
    Context(s): {contexts}
    
    Answer:""")
    
    contexts_text = "\n\n".join(latest_context["contexts"])
    
    answer = llm.invoke(prompt.format_messages(
        question=latest_question["question"],
        contexts=contexts_text
    )).content.strip()
    
    new_answer = {
        "question_id": latest_question["id"],
        "answer": answer
    }
    
    return {
        "answers": [new_answer]
    }

##### Helper methods for analytics

In [ ]:
def calculate_distribution(questions: List[Dict]) -> Dict:
    """Calculate evolution type distribution"""
    distribution = Counter(q["evolution_type"] for q in questions)
    total = len(questions)
    
    return {
        "evolution_type_distribution": dict(distribution),
        "evolution_type_percentages": {
            evo_type: (count / total) * 100 
            for evo_type, count in distribution.items()
        },
        "total_questions": total
    }

def display_results(result: Dict):
    """Display results with analytics"""
    analytics = result['analytics']
    
    print("📊 SYNTHETIC DATA RESULTS")
    print("=" * 30)
    print(f"Total Questions Generated: {analytics['total_questions']}")
    
    print("\n🔄 Evolution Type Distribution:")
    for evo_type, count in analytics['evolution_type_distribution'].items():
        percentage = analytics['evolution_type_percentages'][evo_type]
        print(f"  {evo_type}: {count} ({percentage:.1f}%)")
    
    print("\n📝 Sample Questions:")
    for i, q in enumerate(result['evolved_questions'][:3]):
        answer = next(a for a in result['question_answers'] if a['question_id'] == q['id'])
        print(f"\n{i+1}. [{q['evolution_type'].upper()}]")
        print(f"   Q: {q['question']}")
        print(f"   A: {answer['answer'][:100]}...")

In [31]:
class EvolInstructSDG:
  """LangGraph workflow for synthetic data generation"""

  def __init__(self, llm):
    self.llm = llm
    self.graph = self._build_graph()

  def _build_graph(self):
    """Build the LangGraph workflow"""
    workflow = StateGraph(State)

    workflow.add_node("generate_question", self._generate_question)
    workflow.add_node("generate_answer", self._generate_answer)

    workflow.set_entry_point("generate_question")
    workflow.add_edge("generate_question", "generate_answer")

    workflow.add_conditional_edges(
      "generate_question",
      self._should_continue,
      {
        "continue": "generate_question",
        "finish": END
      }
    )

    return workflow.compile()
  
  def _generate_question(self, state: State) -> State:
    """Generate evolved question based on random evolution type"""
    evolution_types = ["simple", "multi_context", "reasoning"]
    evolution_type = random.choice(evolution_types)
    
    if evolution_type == "simple":
        return simple_evolution(state, self.llm)
    elif evolution_type == "multi_context":
        return multi_context_evolution(state, self.llm)
    else:
        return reasoning_evolution(state, self.llm)
    
  def _generate_answer(self, state: State) -> State:
      """Generate answer using helper function"""
      return generate_answer(state, self.llm)
  
  def _should_continue(self, state: State) -> str:
      if state["current_count"] >= state["target_count"]:
          return "finish"
      return "continue"
  
  def generate(self, documents: List[Document], target_count: int = 5):
    """Generate synthetic data"""
    
    initial_state = {
        "documents": documents,
        "target_count": target_count,
        "current_count": 0,
        "llm": self.llm,
    }
    
    final_state = self.graph.invoke(initial_state)
    analytics = calculate_distribution(final_state["questions"])
    
    return {
        "evolved_questions": final_state["questions"],
        "question_answers": final_state["answers"],
        "question_contexts": final_state["contexts"],
        "analytics": analytics
    }

### Results

In [32]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [33]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
path = "./data"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
sdg = EvolInstructSDG(llm)

result = sdg.generate(documents=docs[:5], target_count=6)
display_results(result)

📊 SYNTHETIC DATA RESULTS
Total Questions Generated: 6

🔄 Evolution Type Distribution:
  reasoning: 2 (33.3%)
  simple: 2 (33.3%)
  multi_context: 2 (33.3%)

📝 Sample Questions:

1. [REASONING]
   Q: How does the classification of an academic calendar as subscription-based affect the requirements for determining full-time enrollment and the awarding of Title IV financial aid?
   A: The classification of an academic calendar as subscription-based affects the requirements for determ...

2. [SIMPLE]
   Q: What are the major changes in Volume 3 of the FSA Handbook for the 2025-2026 academic year?
   A: For the 2025-2026 academic year, there are no major changes in Volume 3 of the FSA Handbook. However...

3. [MULTI_CONTEXT]
   Q: How do the definitions and requirements for determining a student's cost of attendance in Volume 3 relate to the expectations for academic engagement in asynchronous coursework as outlined in Context 2?
   A: The definitions and requirements for determining a stude